# 🛸 SpaceRAG AI — GS2026.1 · FIAP 2TIAPF
> **Assistente Inteligente com RAG para Dados e Documentos Espaciais**
>
> Execute cada célula em ordem (▶ ou Shift+Enter)

| Etapa | Célula | Descrição |
|---|---|---|
| 1 | Instalar dependências | Instala todos os pacotes necessários |
| 2 | Configurar API Key | Carrega a chave da Anthropic |
| 3 | Criar arquivos do projeto | Gera toda a estrutura de código |
| 4 | Rodar a aplicação | Inicia o Streamlit no Colab |
| 5 | Documento de teste (opcional) | Cria um TXT de exemplo para testar |


## 📦 Célula 1 — Instalar dependências

In [1]:
%%capture
!pip install streamlit==1.35.0 \
            anthropic \
            langchain==0.2.16 \
            langchain-core==0.2.38 \
            langchain-community==0.2.16 \
            langchain-text-splitters==0.2.4 \
            chromadb==0.5.3 \
            PyPDF2==3.0.1 \
            python-docx==1.1.2 \
            numpy \
            sentence-transformers

print("✅ Dependências instaladas!")
print("⚠️  Faça Runtime → Restart session antes de continuar!")


> ⚠️ **IMPORTANTE após a Célula 1:** Vá em **Runtime → Restart session** (ou pressione `Ctrl+M .`) antes de executar as próximas células. Isso garante que os pacotes instalados sejam carregados corretamente pelo Streamlit.


## 🔑 Célula 2 — Configurar API Key

**Opção A (recomendada):** Clique em 🔑 na barra lateral → *Add new secret* → `ANTHROPIC_API_KEY`

**Opção B:** Cole a chave em `CHAVE_MANUAL` abaixo (menos seguro)


In [2]:
import os
from google.colab import userdata

CHAVE_MANUAL = 'sk-ant-api03-h0i...1wAA'  # ← cole aqui somente se não usar Secrets (ex: 'sk-ant-...')

try:
    chave = userdata.get('ANTHROPIC_API_KEY')
    if not chave:
        raise ValueError('Vazia')
    os.environ['ANTHROPIC_API_KEY'] = chave
    print('✅ API Key carregada dos Secrets do Colab')
except Exception:
    if CHAVE_MANUAL and CHAVE_MANUAL.strip().startswith('sk-ant-'):
        os.environ['ANTHROPIC_API_KEY'] = CHAVE_MANUAL.strip()
        print('✅ API Key carregada manualmente')
    else:
        print('❌ API Key não configurada!')
        print('   → Adicione nos Secrets do Colab (🔑) ou preencha CHAVE_MANUAL acima')


✅ API Key carregada manualmente


## 📁 Célula 3 — Criar estrutura de pastas e arquivos

In [3]:
import os
os.makedirs('/content/spacerag/src', exist_ok=True)
os.makedirs('/content/spacerag/data/documentos', exist_ok=True)
os.makedirs('/content/spacerag/vector_db', exist_ok=True)
print('✅ Pastas criadas')


✅ Pastas criadas


In [4]:
%%writefile /content/spacerag/src/__init__.py
# SpaceRAG AI — módulos do pipeline RAG


Overwriting /content/spacerag/src/__init__.py


In [5]:
%%writefile /content/spacerag/src/loaders.py
import re
from pathlib import Path
from typing import List
from langchain_core.documents import Document

class DocumentLoader:
    def load(self, file_path: str, file_name: str) -> List[Document]:
        ext = Path(file_name).suffix.lower()
        if ext == '.pdf':    return self._load_pdf(file_path, file_name)
        elif ext == '.docx': return self._load_docx(file_path, file_name)
        elif ext == '.txt':  return self._load_txt(file_path, file_name)
        else: raise ValueError(f'Formato não suportado: {ext}')

    def _load_pdf(self, path, name):
        import PyPDF2
        docs = []
        with open(path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for i, page in enumerate(reader.pages):
                text = self._clean(page.extract_text() or '')
                if text.strip():
                    docs.append(Document(page_content=text,
                        metadata={'source': name, 'page': i+1, 'file_type': 'pdf'}))
        return docs

    def _load_docx(self, path, name):
        from docx import Document as D
        doc = D(path)
        text = self._clean('\n'.join(p.text for p in doc.paragraphs if p.text.strip()))
        return [Document(page_content=text, metadata={'source': name, 'page': 1, 'file_type': 'docx'})]

    def _load_txt(self, path, name):
        for enc in ['utf-8', 'latin-1', 'cp1252']:
            try:
                text = self._clean(open(path, encoding=enc).read())
                return [Document(page_content=text, metadata={'source': name, 'page': 1, 'file_type': 'txt'})]
            except UnicodeDecodeError:
                continue
        raise ValueError(f'Não foi possível decodificar: {name}')

    def _clean(self, text):
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
        text = re.sub(r' {2,}', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()


Overwriting /content/spacerag/src/loaders.py


In [6]:
%%writefile /content/spacerag/src/chunking.py
from typing import List
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

class TextChunker:
    def __init__(self, chunk_size=500, chunk_overlap=50):
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap,
            separators=['\n\n', '\n', '. ', ', ', ' ', ''],
            keep_separator=True, add_start_index=True
        )

    def split(self, documents: List[Document]) -> List[Document]:
        chunks = self._splitter.split_documents(documents)
        for i, c in enumerate(chunks):
            c.metadata['chunk_index'] = i
            c.metadata['chunk_id'] = f"{c.metadata.get('source','?')}::chunk_{i}"
        return chunks


Overwriting /content/spacerag/src/chunking.py


In [7]:
%%writefile /content/spacerag/src/embeddings.py
from typing import List
from langchain_core.embeddings import Embeddings

class LocalEmbeddings(Embeddings):
    """Embeddings usando sentence-transformers local (all-MiniLM-L6-v2).
    Gratuito, sem dependência de API externa para embeddings."""

    MODEL = 'all-MiniLM-L6-v2'

    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self._model = SentenceTransformer(self.MODEL)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._model.encode(texts, show_progress_bar=False).tolist()

    def embed_query(self, text: str) -> List[float]:
        return self._model.encode([text], show_progress_bar=False)[0].tolist()


class EmbeddingModel:
    def __init__(self):
        self._model = LocalEmbeddings()

    def get_langchain_model(self):
        return self._model


Overwriting /content/spacerag/src/embeddings.py


In [8]:
%%writefile /content/spacerag/src/vector_store.py
import os
import uuid
from typing import List, Tuple
from langchain_core.documents import Document

CHROMA_DIR = '/content/spacerag/vector_db/chroma'
COLLECTION  = 'spacerag_docs'

class VectorStore:
    """Vector store usando chromadb diretamente, sem depender do wrapper LangChain
    (que tenta importar sentence_transformers como fallback)."""

    def __init__(self, embedding_model):
        import chromadb
        self._embed_fn = embedding_model.get_langchain_model()
        os.makedirs(CHROMA_DIR, exist_ok=True)
        self._client = chromadb.PersistentClient(path=CHROMA_DIR)
        # Recria a collection a cada processamento para evitar conflitos de IDs
        try:
            self._client.delete_collection(COLLECTION)
        except Exception:
            pass
        self._col = self._client.create_collection(
            name=COLLECTION,
            metadata={'hnsw:space': 'cosine'}
        )
        self._docs: List[Document] = []

    def add_documents(self, chunks: List[Document]):
        self._docs = chunks
        texts = [c.page_content for c in chunks]
        ids   = [c.metadata.get('chunk_id', str(uuid.uuid4())) for c in chunks]
        metas = [c.metadata for c in chunks]

        # Gera embeddings em batch via nossa classe (Voyage API)
        embeddings = self._embed_fn.embed_documents(texts)

        batch_size = 64
        for start in range(0, len(texts), batch_size):
            self._col.add(
                ids=ids[start:start+batch_size],
                embeddings=embeddings[start:start+batch_size],
                documents=texts[start:start+batch_size],
                metadatas=metas[start:start+batch_size],
            )

    def similarity_search_with_score(self, query: str, k: int = 4):
        if self._col.count() == 0:
            raise RuntimeError('Vector store vazio. Processe documentos primeiro.')
        q_emb = self._embed_fn.embed_query(query)
        results = self._col.query(
            query_embeddings=[q_emb],
            n_results=min(k, self._col.count()),
            include=['documents', 'metadatas', 'distances']
        )
        out = []
        for text, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        ):
            doc   = Document(page_content=text, metadata=meta)
            score = 1.0 / (1.0 + float(dist))
            out.append((doc, score))
        return sorted(out, key=lambda x: x[1], reverse=True)

    def count(self):
        try: return self._col.count()
        except: return 0


Overwriting /content/spacerag/src/vector_store.py


In [9]:
%%writefile /content/spacerag/src/llm.py
import os
from typing import List, Dict

ASSISTANT_NAME   = 'SpaceRAG AI'
ASSISTANT_AVATAR = '🛸'

SYSTEM = """Você é o SpaceRAG AI, assistente especializado na Nova Economia Espacial,
desenvolvido para o GS2026.1 da FIAP 2TIAPF. Utiliza a API da Anthropic (Claude).

INSTRUÇÕES:
1. Responda SEMPRE em português brasileiro.
2. Baseie sua resposta EXCLUSIVAMENTE no contexto entre <contexto></contexto>.
3. Se a informação não estiver no contexto, diga: 'Não encontrei essa informação nos documentos carregados.'
4. Cite a fonte (nome do documento e página) quando disponível.
5. Seja técnico mas acessível.
6. Nunca invente dados, estatísticas ou informações."""

class LLMClient:
    MODEL = 'claude-sonnet-4-20250514'

    def __init__(self):
        self.api_key = os.environ.get('ANTHROPIC_API_KEY', '')
        self.name    = ASSISTANT_NAME
        self.avatar  = ASSISTANT_AVATAR

    def generate(self, question: str, context: str, chat_history: List[Dict] = None) -> str:
        import anthropic
        if not self.api_key:
            raise ValueError('ANTHROPIC_API_KEY não configurada. Configure nos Secrets do Colab (🔑).')
        client = anthropic.Anthropic(api_key=self.api_key)
        messages = []
        if chat_history:
            for m in chat_history[-6:]:
                if m['role'] in ('user', 'assistant'):
                    messages.append({'role': m['role'], 'content': m['content']})
        messages.append({
            'role': 'user',
            'content': f'<contexto>\n{context}\n</contexto>\n\nPergunta: {question}'
        })
        resp = client.messages.create(
            model=self.MODEL,
            max_tokens=1500,
            system=SYSTEM,
            messages=messages
        )
        return resp.content[0].text


Overwriting /content/spacerag/src/llm.py


In [10]:
%%writefile /content/spacerag/src/rag_pipeline.py
from typing import List, Dict, Any

TMPL = """
--- Trecho {idx} ---
Fonte: {source} | Página: {page} | Score: {score:.1%}
{text}
"""

class RAGPipeline:
    def __init__(self, vector_store, llm, top_k=4):
        self.vs     = vector_store
        self.llm    = llm
        self.top_k  = top_k

    def query(self, question: str, chat_history: List[Dict] = None) -> Dict[str, Any]:
        results = self.vs.similarity_search_with_score(question, k=self.top_k)
        context_parts, sources = [], []
        for idx, (doc, score) in enumerate(results, 1):
            source = doc.metadata.get('source', '?')
            page   = doc.metadata.get('page', '?')
            context_parts.append(TMPL.format(
                idx=idx, source=source, page=page, score=score, text=doc.page_content))
            sources.append({'source': source, 'page': page, 'score': score, 'text': doc.page_content})
        context = '\n'.join(context_parts)
        answer  = self.llm.generate(question, context, chat_history)
        return {'answer': answer, 'sources': sources}


Overwriting /content/spacerag/src/rag_pipeline.py


In [11]:
%%writefile /content/spacerag/src/utils.py
import json, os
HISTORY_FILE = '/content/spacerag/data/chat_history.json'

def save_chat_history(history):
    os.makedirs(os.path.dirname(HISTORY_FILE), exist_ok=True)
    serializable = []
    for m in history:
        e = {'role': m['role'], 'content': m['content']}
        if 'sources' in m:
            e['sources'] = [
                {'source': s.get('source'), 'page': s.get('page'),
                 'score': float(s.get('score', 0)), 'text': s.get('text', '')[:200]}
                for s in m['sources']
            ]
        serializable.append(e)
    json.dump(serializable, open(HISTORY_FILE, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

def load_chat_history():
    if not os.path.exists(HISTORY_FILE): return []
    try: return json.load(open(HISTORY_FILE, encoding='utf-8'))
    except: return []


Overwriting /content/spacerag/src/utils.py


In [12]:
%%writefile /content/spacerag/app.py
import streamlit as st
import os, sys
from pathlib import Path
sys.path.insert(0, '/content/spacerag')

from src.loaders      import DocumentLoader
from src.chunking     import TextChunker
from src.embeddings   import EmbeddingModel
from src.vector_store import VectorStore
from src.rag_pipeline import RAGPipeline
from src.llm          import LLMClient
from src.utils        import save_chat_history, load_chat_history

llm_info         = LLMClient()
ASSISTANT_NAME   = llm_info.name
ASSISTANT_AVATAR = llm_info.avatar

st.set_page_config(
    page_title=f'SpaceRAG AI · GS2026.1',
    page_icon='🛸',
    layout='wide'
)

st.markdown("""
<style>
.badge {
    display:inline-flex; align-items:center; gap:6px;
    background:#1a3a5c22; border:1px solid #1a3a5c55;
    border-radius:20px; padding:3px 14px;
    font-size:13px; font-weight:600; color:#1a5ca0;
    margin-bottom:8px;
}
.source-chunk {
    background:#f0f6ff; border-left:4px solid #1a5ca0;
    padding:0.7rem; border-radius:0 8px 8px 0;
    margin:0.4rem 0; font-size:0.83rem;
}
.score-badge {
    background:#1a5ca0; color:white;
    padding:1px 8px; border-radius:10px; font-size:0.75rem;
}
</style>
""", unsafe_allow_html=True)

# ── Header
col_logo, col_title = st.columns([1, 8])
with col_logo:
    st.markdown('# 🛸')
with col_title:
    st.markdown('# SpaceRAG AI')
    st.markdown(
        '<div class="badge">🤖 Powered by Claude · Anthropic</div>',
        unsafe_allow_html=True
    )
st.caption('Assistente RAG para a Nova Economia Espacial · GS2026.1 FIAP 2TIAPF')
st.divider()

# ── Session state
for key, val in [
    ('vector_store', None),
    ('rag_pipeline', None),
    ('chat_history', load_chat_history()),
    ('doc_stats', {'docs': 0, 'chunks': 0, 'store': 0}),
    ('show_transp', False),
]:
    if key not in st.session_state:
        st.session_state[key] = val

# ── Sidebar
with st.sidebar:
    st.markdown('<div class="badge">🛸 SpaceRAG AI · GS2026.1</div>', unsafe_allow_html=True)
    st.header('⚙️ Configurações')

    api_key = st.text_input(
        '🔑 Anthropic API Key',
        type='password',
        placeholder='✅ Configurada via Secrets' if os.environ.get('ANTHROPIC_API_KEY') else 'sk-ant-...',
        help='Configure nos Secrets do Colab (🔑) ou cole aqui'
    )
    if api_key:
        os.environ['ANTHROPIC_API_KEY'] = api_key
        st.success('✅ API Key configurada!')
    elif os.environ.get('ANTHROPIC_API_KEY'):
        st.success('✅ API Key ativa (via Secrets)')
    else:
        st.error('❌ API Key não configurada')

    st.divider()
    st.subheader('📂 Documentos')
    files = st.file_uploader(
        'Upload PDF, TXT ou DOCX',
        type=['pdf', 'txt', 'docx'],
        accept_multiple_files=True
    )
    chunk_size    = st.slider('Tamanho do chunk',  200, 1000, 500, 50)
    chunk_overlap = st.slider('Sobreposição',        0,  200,  50, 10)
    top_k         = st.slider('Top-k chunks',        1,   10,   4)
    st.session_state.show_transp = st.toggle(
        '🔍 Modo Transparente RAG',
        st.session_state.show_transp,
        help='Exibe os trechos recuperados e seus scores de similaridade'
    )
    process = st.button(
        '🚀 Processar Documentos',
        type='primary',
        use_container_width=True,
        disabled=not files
    )
    if st.button('🗑️ Limpar Histórico', use_container_width=True):
        st.session_state.chat_history = []
        save_chat_history([])
        st.rerun()

    st.divider()
    st.subheader('📊 Status')
    c1, c2 = st.columns(2)
    c1.metric('📄 Docs',    st.session_state.doc_stats['docs'])
    c1.metric('🧩 Chunks',  st.session_state.doc_stats['chunks'])
    c2.metric('💾 Vetores', st.session_state.doc_stats['store'])
    c2.metric('🤖 Modelo',  'Claude Sonnet 4')

    with st.expander('ℹ️ Arquitetura'):
        st.markdown("""
        **LLM:** Claude (Anthropic)
        `claude-sonnet-4-20250514`

        **Embeddings:** Voyage (Anthropic API)
        `all-MiniLM-L6-v2`

        **Vector Store:** ChromaDB

        **Framework:** LangChain

        **Interface:** Streamlit
        """)

# ── Processamento de documentos
if process and files:
    with st.status('⚙️ Processando documentos...', expanded=True) as status:
        try:
            loader   = DocumentLoader()
            all_docs = []
            for f in files:
                p = Path(f'/content/spacerag/data/documentos/{f.name}')
                p.parent.mkdir(parents=True, exist_ok=True)
                p.write_bytes(f.getbuffer())
                docs = loader.load(str(p), f.name)
                all_docs.extend(docs)
                st.write(f'  📄 {f.name} → {len(docs)} página(s)')
            st.write(f'✅ {len(files)} documento(s) carregado(s) — {len(all_docs)} páginas no total')

            chunks = TextChunker(chunk_size, chunk_overlap).split(all_docs)
            st.write(f'✅ {len(chunks)} chunks gerados')

            st.write('⚙️ Gerando embeddings (pode levar ~30s na primeira vez)...')
            em = EmbeddingModel()
            vs = VectorStore(em)
            vs.add_documents(chunks)
            st.session_state.vector_store = vs
            st.session_state.rag_pipeline = RAGPipeline(vs, LLMClient(), top_k)
            st.session_state.doc_stats    = {
                'docs': len(files), 'chunks': len(chunks), 'store': vs.count()
            }
            status.update(label='✅ Base vetorial pronta! Pode fazer perguntas.', state='complete')
        except Exception as e:
            status.update(label=f'❌ Erro no processamento', state='error')
            st.error(str(e))

# ── Chat
st.subheader(f'💬 Chat com o {ASSISTANT_NAME}')

if st.session_state.rag_pipeline is None:
    st.info('👈 Faça upload de documentos na barra lateral e clique em **Processar Documentos** para ativar o chat.')

for msg in st.session_state.chat_history:
    avatar = '🧑' if msg['role'] == 'user' else ASSISTANT_AVATAR
    with st.chat_message(msg['role'], avatar=avatar):
        st.markdown(msg['content'])
        if msg['role'] == 'assistant' and 'sources' in msg and st.session_state.show_transp:
            with st.expander('🔍 Trechos recuperados pelo RAG'):
                for i, s in enumerate(msg['sources'], 1):
                    st.markdown(
                        f'<div class="source-chunk">'
                        f'<b>Chunk {i}</b> — '
                        f'<span class="score-badge">{s["score"]:.1%}</span> — '
                        f'<i>{s["source"]}</i> pág.{s["page"]}<br><br>'
                        f'{s["text"][:350]}'
                        f'</div>',
                        unsafe_allow_html=True
                    )

if prompt := st.chat_input(
    'Pergunte sobre dados espaciais, satélites, clima, agricultura...',
    disabled=st.session_state.rag_pipeline is None
):
    st.session_state.chat_history.append({'role': 'user', 'content': prompt})
    with st.chat_message('user', avatar='🧑'):
        st.markdown(prompt)

    with st.chat_message('assistant', avatar=ASSISTANT_AVATAR):
        with st.spinner(f'{ASSISTANT_NAME} está consultando os documentos...'):
            try:
                result = st.session_state.rag_pipeline.query(
                    prompt, st.session_state.chat_history[:-1]
                )
                st.markdown(result['answer'])

                if st.session_state.show_transp and result['sources']:
                    with st.expander('🔍 Trechos recuperados pelo RAG'):
                        for i, s in enumerate(result['sources'], 1):
                            st.markdown(
                                f'<div class="source-chunk">'
                                f'<b>Chunk {i}</b> — '
                                f'<span class="score-badge">{s["score"]:.1%}</span> — '
                                f'<i>{s["source"]}</i> pág.{s["page"]}<br><br>'
                                f'{s["text"][:350]}'
                                f'</div>',
                                unsafe_allow_html=True
                            )

                st.session_state.chat_history.append({
                    'role':    'assistant',
                    'content': result['answer'],
                    'sources': result['sources']
                })
                save_chat_history(st.session_state.chat_history)

            except Exception as e:
                st.error(f'Erro ao gerar resposta: {e}')

print('✅ app.py gerado com sucesso!')


Overwriting /content/spacerag/app.py


## 🌐 Célula 4 — Rodar a aplicação

> Execute esta célula por último. O link gerado funciona enquanto a célula estiver ativa.


In [16]:
import subprocess, time, os
from google.colab.output import eval_js

# Mata qualquer Streamlit anterior
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

PORT = 8501
env = os.environ.copy()

proc = subprocess.Popen(
    ['streamlit', 'run', '/content/spacerag/app.py',
     '--server.port', str(PORT),
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=env
)

time.sleep(6)
url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
print('🛸 SpaceRAG AI rodando em:', url)

🛸 SpaceRAG AI rodando em: https://8501-m-s-kkb-usw4a1-23i7iu3uxi0nm-a.us-west4-1.prod.colab.dev


## 📄 Célula 5 (opcional) — Criar documento de teste

Se não tiver um PDF/TXT próprio, esta célula gera um documento de exemplo sobre a Nova Economia Espacial para testar o sistema.


In [14]:
doc_teste = """Nova Economia Espacial — Base de Conhecimento
================================================

1. SENSORIAMENTO REMOTO E SATÉLITES
O sensoriamento remoto por satélite revolucionou o monitoramento ambiental. Satélites como
Landsat-9 (NASA/USGS) e Sentinel-2 (ESA) oferecem imagens multiespectrais com resolução
de 10 a 30 metros, permitindo análise de vegetação, solo, água e atmosfera.

O índice NDVI (Normalized Difference Vegetation Index) é calculado a partir das bandas
do infravermelho próximo e do vermelho: NDVI = (NIR - RED) / (NIR + RED). Valores acima
de 0,4 indicam vegetação densa; abaixo de 0,2 indicam solo exposto ou vegetação degradada.

2. MONITORAMENTO DE DESMATAMENTO
O sistema PRODES (INPE) monitora o desmatamento na Amazônia Legal com precisão de ~95%,
utilizando imagens Landsat. Em 2023, o desmatamento na Amazônia caiu 50% em relação ao
ano anterior, atingindo 11.568 km² segundo o INPE.

O DETER (Detecção de Desmatamento em Tempo Real) gera alertas em até 24h usando imagens
MODIS de 250m de resolução, complementando o PRODES com maior agilidade.

3. AGRICULTURA DE PRECISÃO ESPACIAL
Satélites de observação da Terra permitem agricultura de precisão em escala global.
A constelação Planet Labs possui mais de 200 satélites CubeSat que revisitam qualquer
ponto do globo diariamente com resolução de 3 metros.

Técnicas como NDWI (Normalized Difference Water Index) monitoram umidade do solo,
essencial para irrigação eficiente. Estimativas indicam que a agricultura de precisão
pode reduzir o uso de água em até 30% e fertilizantes em 20%.

4. MUDANÇAS CLIMÁTICAS E SATÉLITES
O satélite OCO-2 (Orbiting Carbon Observatory) da NASA mede concentrações atmosféricas
de CO2 com precisão de 1-2 ppm. Em 2023, a concentração média global ultrapassou 420 ppm,
o maior nível dos últimos 3 milhões de anos.

O GRACE-FO (Gravity Recovery and Climate Experiment Follow-On) mede variações na
massa de água doce global, revelando aceleração no derretimento de geleiras. O
gelo da Groenlândia perde cerca de 280 bilhões de toneladas por ano.

5. DESASTRES NATURAIS
O Copernicus Emergency Management Service (CEMS) da União Europeia ativa mapeamento
rápido por satélite em desastres naturais dentro de 12 horas. Desde 2012, o serviço
atendeu mais de 700 ativações emergenciais em 100+ países.

Satélites SAR (Synthetic Aperture Radar) como o Sentinel-1 operam independente de
nuvens ou horário, sendo essenciais para monitorar inundações e deslizamentos.

6. NOVA ECONOMIA ESPACIAL
O mercado global do setor espacial atingiu US$ 546 bilhões em 2023, com previsão de
alcançar US$ 1 trilhão até 2030 (Morgan Stanley). O custo de lançamento caiu de
US$ 65.000/kg (Space Shuttle) para menos de US$ 2.700/kg (Falcon 9 reutilizável).

A constelação Starlink da SpaceX conta com mais de 5.000 satélites em órbita baixa
(LEO), fornecendo internet de alta velocidade para regiões remotas. Outras constelações
como OneWeb e Amazon Kuiper buscam fatias do mesmo mercado.

7. BRASIL NO ESPAÇO
O Brasil opera o CBERS (China-Brazil Earth Resources Satellite), fruto de parceria
sino-brasileira desde 1988. O CBERS-4A, lançado em 2019, possui câmera de 2 metros
de resolução e gera imagens gratuitas para pesquisadores.

O Centro Espacial de Alcântara (CEA) no Maranhão é considerado um dos melhores
pontos de lançamento do mundo pela proximidade com a linha do Equador, reduzindo
o consumo de combustível em até 30% comparado a outros centros.
"""

path = '/content/spacerag/data/documentos/nova_economia_espacial.txt'
with open(path, 'w', encoding='utf-8') as f:
    f.write(doc_teste)

print(f'✅ Documento criado: {path}')
print(f'   Tamanho: {len(doc_teste):,} caracteres')
print()
print('👉 Na interface do SpaceRAG:')
print('   1. Clique em "Browse files" na sidebar')
print('   2. Selecione o arquivo acima')
print('   3. Clique em "Processar Documentos"')
print()
print('💡 Perguntas de teste:')
print('   - O que é NDVI e como é calculado?')
print('   - Qual satélite monitora CO2 atmosférico?')
print('   - Quantos satélites tem a constelação Starlink?')
print('   - Como o Copernicus responde a desastres naturais?')
print('   - Qual a precisão do PRODES para detectar desmatamento?')


✅ Documento criado: /content/spacerag/data/documentos/nova_economia_espacial.txt
   Tamanho: 3,443 caracteres

👉 Na interface do SpaceRAG:
   1. Clique em "Browse files" na sidebar
   2. Selecione o arquivo acima
   3. Clique em "Processar Documentos"

💡 Perguntas de teste:
   - O que é NDVI e como é calculado?
   - Qual satélite monitora CO2 atmosférico?
   - Quantos satélites tem a constelação Starlink?
   - Como o Copernicus responde a desastres naturais?
   - Qual a precisão do PRODES para detectar desmatamento?


---
## ℹ️ Perguntas de teste sugeridas

Após carregar o documento de exemplo (Célula 5), experimente:

- *O que é NDVI e como é calculado?*
- *Qual satélite monitora CO2 atmosférico?*
- *Quantos satélites tem a constelação Starlink?*
- *Como o Copernicus responde a desastres naturais?*
- *Qual a precisão do PRODES para detectar desmatamento?*
- *O que é o GRACE-FO e o que ele mede?*
- *Qual o custo de lançamento do Falcon 9?*
- *Qual a vantagem do Centro Espacial de Alcântara?*

## 🏗️ Arquitetura da Solução

```
PDF/TXT/DOCX
    ↓ DocumentLoader (PyPDF2 / python-docx)
    ↓ TextChunker (RecursiveCharacterTextSplitter)
    ↓ EmbeddingModel (sentence-transformers/all-MiniLM-L6-v2)
    ↓ VectorStore (ChromaDB)
         ↓ similarity_search (pergunta do usuário)
         ↓ contexto recuperado (top-k chunks)
    ↓ LLMClient (Claude claude-sonnet-4-20250514)
    ↓ Resposta em português
    ↓ Interface Streamlit
```

## ✅ Requisitos atendidos

| Requisito | Implementação |
|---|---|
| Interface simples | Streamlit com chat nativo |
| Upload de documentos | PDF, TXT e DOCX |
| Geração de embeddings | sentence-transformers/all-MiniLM-L6-v2 |
| Vector store | ChromaDB com persistência |
| Modelo generativo | Claude (Anthropic) via API |
| Demonstração funcional | Documento de teste incluído |
